### Ячейка 1: Импорты и Основные Настройки

In [12]:
### Ячейка 1: Импорты и Основные Настройки
# -*- coding: utf-8 -*-
# Ячейка 1: Импорты и Основные Настройки (v1.1 - Добавлены math, traceback)

import os
import gc
import sys
import json
import re
import warnings
import math # <--- Добавлен импорт
import traceback # <--- Добавлен импорт
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from collections import Counter, defaultdict, OrderedDict
os.environ['KMP_DUPLICATE_LIB_OK']='True'
# Подавление стандартных предупреждений (опционально)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Стандартные библиотеки и ML/Data Science
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import Levenshtein # pip install python-Levenshtein

# PyTorch и связанные библиотеки
import torch
import torch.nn as nn
import torch.nn.functional as F
# random_split импортируем здесь же, так как он нужен для разделения данных
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
import torchaudio # Может понадобиться для SpecAugment в MorseDataset

# Утилиты и Визуализация
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# --- Основные Настройки ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {DEVICE}")

# Установим стиль для графиков
plt.style.use('seaborn-v0_8-darkgrid')
# Увеличим размер шрифта по умолчанию для читаемости
plt.rcParams.update({'font.size': 12})

# Очистка памяти GPU (на всякий случай)
if DEVICE == torch.device('cuda'):
    torch.cuda.empty_cache()
    gc.collect()

print("Импорты и базовые настройки завершены.")

Используемое устройство: cuda
Импорты и базовые настройки завершены.


### Ячейка 2: Определение Классов Модели, Dataset и Collate Fn


In [13]:
# Ячейка 2: Определение Классов Модели, Dataset и Collate Fn
# ВАЖНО: Убедитесь, что эти определения соответствуют тем,
# которые использовались при обучении моделей, которые вы будете загружать!

# --- Вспомогательные классы для модели ---
class SELayer(nn.Module):
    def __init__(self, channel: int, reduction: int = 16):
        super().__init__()
        if channel <= reduction: reduction = channel // 2 if channel > 1 else 1
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channel, channel // reduction, kernel_size=1, bias=False),
            nn.GELU(),
            nn.Conv2d(channel // reduction, channel, kernel_size=1, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.avg_pool(x); y = self.fc(y); return x * y.expand_as(x)

class ResBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int,
                 kernel_size: Tuple[int, int], stride: Tuple[int, int] = (1, 1),
                 use_se: bool = True, se_reduction_ratio: int = 16):
        super().__init__()
        padding = (kernel_size[0] // 2, kernel_size[1] // 2)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels); self.activation1 = nn.GELU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size, stride=1, padding=padding, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.se = SELayer(out_channels, se_reduction_ratio) if use_se else nn.Identity()
        self.shortcut = nn.Sequential()
        if stride != (1, 1) or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        self.final_activation = nn.GELU()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.shortcut(x); out = self.conv1(x); out = self.bn1(out); out = self.activation1(out)
        out = self.conv2(out); out = self.bn2(out); out = self.se(out); out += identity; out = self.final_activation(out)
        return out

# --- Основная модель ---
class MorseRecognizer(nn.Module):
    def __init__(self, config: dict):
        super().__init__()
        model_cfg = config["model"]
        input_freq_dim = model_cfg.get("freq_dim"); vocab_size = model_cfg.get("vocab_size")
        if input_freq_dim is None or vocab_size is None: raise ValueError("freq_dim или vocab_size отсутствуют в конфиге модели!")
        input_channels = 1
        cnn_block_channels = model_cfg.get("cnn_block_channels", [32, 64]); cnn_kernels = model_cfg.get("cnn_kernel_size", [[3, 5], [3, 5]])
        cnn_strides = model_cfg.get("cnn_stride", [[2, 2], [2, 2]]); use_se = model_cfg.get("cnn_use_se", True)
        se_reduction = model_cfg.get("cnn_se_reduction_ratio", 16); num_cnn_blocks = len(cnn_block_channels)
        if not (len(cnn_kernels) == num_cnn_blocks and len(cnn_strides) == num_cnn_blocks): raise ValueError("Несоответствие длин параметров CNN!")
        cnn_layers = []; in_ch = input_channels; self._time_reduction_factor = 1; self._freq_reduction_factor = 1; current_freq_dim = input_freq_dim
        for i in range(num_cnn_blocks):
            out_ch = cnn_block_channels[i]; kernel = cnn_kernels[i]; stride = cnn_strides[i]
            cnn_layers.append(ResBlock(in_ch, out_ch, kernel, stride, use_se, se_reduction))
            in_ch = out_ch; self._freq_reduction_factor *= stride[0]; self._time_reduction_factor *= stride[1]
            current_freq_dim = math.ceil(current_freq_dim / stride[0])
        self.cnn = nn.Sequential(*cnn_layers)
        rnn_input_size = 0
        try:
            self.eval(); dummy_input_T = max(100, self._time_reduction_factor * 4)
            with torch.no_grad(): dummy_output = self.cnn(torch.randn(1, input_channels, dummy_input_T, input_freq_dim))
            rnn_input_size = dummy_output.shape[1] * dummy_output.shape[3]
            self.train()
        except Exception as e: print(f"Ошибка dummy forward: {e}"); rnn_input_size = model_cfg["cnn_block_channels"][-1] * math.ceil(input_freq_dim / self._freq_reduction_factor)
        if rnn_input_size <= 0: raise ValueError(f"Некорректный rnn_input_size: {rnn_input_size}")
        self.rnn = nn.GRU(rnn_input_size, model_cfg["rnn_hidden_size"], model_cfg["rnn_layers"], batch_first=True, bidirectional=True, dropout=model_cfg["rnn_dropout"] if model_cfg["rnn_layers"] > 1 else 0.0)
        self.dropout = nn.Dropout(model_cfg["dropout"])
        self.classifier = nn.Linear(model_cfg["rnn_hidden_size"] * 2, vocab_size)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.cnn(x); B, C_out, T_reduced, F_reduced = x.shape; x = x.permute(0, 2, 1, 3); x = x.reshape(B, T_reduced, C_out * F_reduced)
        x, _ = self.rnn(x); x = self.dropout(x); x = self.classifier(x); return x
    def get_time_reduction_factor(self) -> int: return max(1, self._time_reduction_factor)

# --- Функции и классы для данных ---
def create_full_path(filename, folder_path: Path) -> Path:
    return folder_path / Path(str(filename)).with_suffix('.opus')

def get_features(audio_path: Path, audio_config: dict, augmenter=None, apply_audio_aug_prob=0.0, is_train=False) -> Optional[np.ndarray]:
    try:
        if not audio_path.is_file(): return None
        y, current_sr = sf.read(audio_path, dtype='float32')
        target_sr = audio_config["sample_rate"]
        if y is None or y.shape[0] == 0: return None
        if current_sr != target_sr: y = librosa.resample(y=y, orig_sr=current_sr, target_sr=target_sr)
        if y is None or y.shape[0] == 0: return None
        # Аугментации здесь не применяем для инференса
        stft_result = librosa.stft(y=y, n_fft=audio_config["n_fft"], hop_length=audio_config["hop_length"])
        features_ft = np.abs(stft_result).astype(np.float32)
        if np.isnan(features_ft).any() or np.isinf(features_ft).any(): return None
        if features_ft.shape[1] == 0: return None
        return features_ft
    except Exception: return None # Упрощенная обработка ошибок для инференса

class SpecAugmentTransform(nn.Module): # Нужен, если MorseDataset его использует
    def __init__(self, aug_config: dict): super().__init__(); self.apply_spec_augment = False; self.transform = nn.Identity() # Отключен для инференса
    def forward(self, x: torch.Tensor) -> torch.Tensor: return x

class MorseDataset(Dataset):
    def __init__(self, df: pd.DataFrame, char_to_int: Dict[str, int], config: dict, is_train: bool, text_column: str = 'message', **kwargs): # kwargs для совместимости
        self.df = df.copy().reset_index(drop=True)
        self.char_to_int = char_to_int
        self.config = config # Используем конфиг, специфичный для модели
        self.is_train = False # Всегда False для инференса/анализа
        self.text_column = text_column
        self.file_id_column = config.get('train_file_column', config.get('test_file_column', 'id')) # Определяем колонку ID
        self.data_dir = Path(self.config.get("paths", {}).get("data_dir", "."))
        self.audio_folder_name = self.config.get("paths", {}).get("audio_folder_name", "audio")
        self.audio_folder = self.data_dir / self.audio_folder_name
        self.blank_idx = self.config.get("ctc", {}).get("blank_idx", 0) # Получаем blank_idx из конфига

    def __len__(self): return len(self.df)
    def __getitem__(self, idx) -> Optional[Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]]:
        if idx >= len(self.df): return None
        row = self.df.iloc[idx]
        file_id = row.get(self.file_id_column, f"unknown_id_at_index_{idx}")
        morse_text = row.get(self.text_column, "") # Может быть пустым для test_df
        try: audio_path = create_full_path(file_id, self.audio_folder)
        except Exception: return None
        original_spec_ft = get_features(audio_path, self.config["audio"])
        if original_spec_ft is None: return None
        features_ctf = np.expand_dims(original_spec_ft.T, axis=0)
        features_tensor = torch.tensor(features_ctf, dtype=torch.float32)
        if torch.isnan(features_tensor).any() or torch.isinf(features_tensor).any(): return None
        encoded_text_list = []
        if pd.notna(morse_text) and isinstance(morse_text, str): # Кодируем текст, только если он есть (для val_df)
            for char in morse_text:
                idx_ = self.char_to_int.get(char, self.blank_idx) # Заменяем неизвестные на бланк
                encoded_text_list.append(idx_)
        encoded_text = torch.tensor(encoded_text_list, dtype=torch.long)
        input_length = torch.tensor(features_tensor.shape[1], dtype=torch.long)
        target_length = torch.tensor(len(encoded_text), dtype=torch.long)
        if input_length.item() <= 0: return None
        return features_tensor, encoded_text, input_length, target_length

def collate_fn(batch: List[Optional[Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]]], pad_idx: int) -> Optional[Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]]:
    valid_batch = [b for b in batch if b is not None]
    if not valid_batch: return None
    try: features, texts, input_lengths, target_lengths = zip(*valid_batch)
    except ValueError: return None
    if not features: return None
    num_channels = features[0].shape[0]; num_freqs = features[0].shape[2]
    valid_indices = [i for i, f in enumerate(features) if f.dim() == 3 and f.shape[0] == num_channels and f.shape[2] == num_freqs and f.shape[1] > 0]
    if not valid_indices: return None
    valid_features = [features[i] for i in valid_indices]; valid_texts = [texts[i] for i in valid_indices]
    valid_input_lengths = [input_lengths[i] for i in valid_indices]; valid_target_lengths = [target_lengths[i] for i in valid_indices]
    features_permuted = [f.permute(1, 0, 2) for f in valid_features]
    features_padded = pad_sequence(features_permuted, batch_first=False, padding_value=0.0).permute(1, 2, 0, 3)
    texts_padded = pad_sequence(valid_texts, batch_first=True, padding_value=pad_idx)
    input_lengths_tensor = torch.stack(valid_input_lengths); target_lengths_tensor = torch.stack(valid_target_lengths)
    final_batch_size = features_padded.shape[0]
    if not (final_batch_size == texts_padded.shape[0] == input_lengths_tensor.shape[0] == target_lengths_tensor.shape[0]): return None
    return features_padded, texts_padded, input_lengths_tensor, target_lengths_tensor

# --- Функция декодирования ---
def ctc_greedy_decode(logits: torch.Tensor, int_to_char: Dict[int, str], blank_idx: int) -> List[str]:
    decoded_preds = []
    if logits.dim() != 3: return ["ERROR_DECODE_SHAPE"] * (logits.shape[0] if logits.dim() > 0 else 1)
    best_paths = torch.argmax(logits, dim=2)
    for path in best_paths.cpu().numpy():
        path_no_duplicates = [k for i, k in enumerate(path) if i == 0 or k != path[i-1]]
        path_no_blanks = [c for c in path_no_duplicates if c != blank_idx]
        decoded_text = "".join([int_to_char.get(c, '?') for c in path_no_blanks])
        decoded_preds.append(decoded_text)
    return decoded_preds


print("Классы Модели, Dataset, Collate Fn и Декодер определены.")

Классы Модели, Dataset, Collate Fn и Декодер определены.


### Ячейка 3: Конфигурация Анализа и Загрузка Данных/Моделей

In [14]:
### Ячейка 3: Конфигурация Анализа и Загрузка Данных/Моделей
# Ячейка 3: Конфигурация Анализа и Загрузка Данных/Моделей (v1.2 - Исправлено индексирование iloc)

# --- 1. Глобальные Настройки Данных и Параметров ---
# Эти параметры должны быть согласованы с тем, как данные готовились при обучении
DATA_DIR = Path("./") # Укажите путь к папке с train.csv, sample_submission.csv
TRAIN_CSV_FILENAME = "train.csv"
TEST_CSV_FILENAME = "sample_submission.csv" # Или test.csv, если он содержит только ID

MORSE_CODE_COLUMN = 'message' # Имя колонки с текстом Морзе
FILE_ID_COLUMN = 'id'         # Имя колонки с ID файла (одинаково для train и test)

RANDOM_SEED = 42              # Сид для воспроизводимости разделения train/val
VAL_SPLIT_RATIO = 0.1         # Доля данных для валидации (должна совпадать с обучением)

# Параметры CTC (должны совпадать с обучением)
BLANK_CHAR = "<blank>"
PAD_CHAR = "<pad>"
BLANK_IDX = 0
PAD_IDX = -1 # Индекс паддинга, используемый в collate_fn

# --- 2. Определение моделей для анализа ---
# Замените пути на ваши реальные пути к моделям и конфигам
MODELS_TO_ANALYZE = [
    {
        'name': '0.2637', # Дайте модели осмысленное имя
        'model_path': './0.2637\MorseCRNN_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin_Clean_FINETUNE_OneCycleLR_20250421_033740_best_epoch23_lev0.2637.pth', # Пример пути
        'config_path': './0.2637\config_final_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin_Clean.json' # Пример пути
    },
    { 
        'name': '0.2638',

        'model_path': './0.2638\MorseCRNN_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin_Clean_FINETUNE_OneCycleLR_20250421_171419_best_epoch2_lev0.2637.pth', # Пример пути 2
        'config_path': './0.2638\config_initial_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin_Clean.json' # Пример пути 2
    },
    { 
        'name': '0.2657',

        'model_path': './0.2657\MorseCRNN_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin_Clean_FINETUNE_OneCycleLR_20250421_033740_best_epoch13_lev0.2657.pth', # Пример пути 2
        'config_path': './0.2657\config_final_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin_Clean.json' # Пример пути 2
    },
    { 
        'name': '0.3057',

        'model_path': './0.3057\morse_v9.2_FT_K3x5-3x9_CRNN_ResNetSE_K3x5-3x9_Hop64_v9.2_FINETUNE_FINETUNE_OneCycleLR_20250420_091456_best_epoch6_lev0.3057.pth', # Пример пути 2
        'config_path': './0.3057\config_initial_finetune.json' # Пример пути 2
    },
    { 
        'name': '0.2773',

        'model_path': './0.2773\morse_v9.1_FT_K3x5_CRNN_ResNetSE_K3x5-K3x5_Hop96_v9.1_FINETUNE_FINETUNE_OneCycleLR_20250420_210451_best_epoch9_lev0.2773.pth', # Пример пути 2
        'config_path': './0.2773\config_initial_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin.json' # Пример пути 2
    },
    { 
        'name': '0.3133',

        'model_path': './0.3133\morse_v9.1_Asym_CRNN_ResNetSE_AsymKerns_v9.1_TRAIN_ReduceLROnPlateau_20250419_001559_best_epoch12_lev0.3133.pth', # Пример пути 2
        'config_path': './0.3133\config_initial.json' # Пример пути 2
    },
    { 
        'name': '0.2607',

        'model_path': './0.2607\MorseCRNN_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin_Clean_FINETUNE_OneCycleLR_20250421_193716_best_epoch3_lev0.2607.pth', # Пример пути 2
        'config_path': './0.2607\config_initial_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin_Clean.json' # Пример пути 2
    },
    
]

# --- 3. Загрузка DataFrame'ов и Создание Словарей ---
print("\n--- Загрузка данных и создание словарей ---")
try:
    # Загрузка CSV
    train_df_full = pd.read_csv(DATA_DIR / TRAIN_CSV_FILENAME)
    test_df = pd.read_csv(DATA_DIR / TEST_CSV_FILENAME)
    print(f"Загружен {TRAIN_CSV_FILENAME}: {len(train_df_full)} строк")
    print(f"Загружен {TEST_CSV_FILENAME}: {len(test_df)} строк")

    # Проверка наличия колонок
    if MORSE_CODE_COLUMN not in train_df_full.columns:
        raise ValueError(f"Колонка '{MORSE_CODE_COLUMN}' не найдена в {TRAIN_CSV_FILENAME}")
    if FILE_ID_COLUMN not in train_df_full.columns:
        raise ValueError(f"Колонка '{FILE_ID_COLUMN}' не найдена в {TRAIN_CSV_FILENAME}")
    if FILE_ID_COLUMN not in test_df.columns:
        raise ValueError(f"Колонка '{FILE_ID_COLUMN}' не найдена в {TEST_CSV_FILENAME}")

    # Функция создания словарей (использует глобальные BLANK/PAD параметры)
    def create_char_map(texts: List[str], blank_char=BLANK_CHAR, pad_char=PAD_CHAR, blank_idx=BLANK_IDX, pad_idx=PAD_IDX):
        valid_texts = [str(text) for text in texts if pd.notna(text) and text != '']
        unique_chars = set(char for text in valid_texts for char in text)
        # Убираем blank и pad из уникальных символов, если они там случайно оказались
        unique_chars_data = unique_chars - {blank_char, pad_char}
        sorted_chars = sorted(list(unique_chars_data))

        char_to_int = {blank_char: blank_idx}
        current_idx = 0
        for char in sorted_chars:
            if current_idx == blank_idx:
                current_idx += 1 # Пропускаем индекс бланка
            char_to_int[char] = current_idx
            current_idx += 1
        int_to_char = {i: char for char, i in char_to_int.items()}
        vocab_size = len(char_to_int)
        print(f"Словарь создан: {vocab_size} символов (включая бланк).")
        print(f"  Blank: '{blank_char}' ({blank_idx}), Pad для collate: '{pad_char}' ({pad_idx})")
        return char_to_int, int_to_char, vocab_size

    # Создание словарей на основе ПОЛНОГО train датасета
    char_to_int, int_to_char, vocab_size = create_char_map(train_df_full[MORSE_CODE_COLUMN].tolist())

    # Разделение train_df_full на train_split_df и val_split_df
    if not (0 < VAL_SPLIT_RATIO < 1):
        raise ValueError(f"Некорректный VAL_SPLIT_RATIO ({VAL_SPLIT_RATIO}). Должен быть между 0 и 1.")
    val_size = int(len(train_df_full) * VAL_SPLIT_RATIO)
    train_size = len(train_df_full) - val_size
    if val_size <= 0 or train_size <= 0:
        raise ValueError(f"Некорректные размеры train/val: Train={train_size}, Val={val_size}.")

    print(f"Разделение {len(train_df_full)} записей на Train ({train_size}) и Val ({val_size}) с Seed: {RANDOM_SEED}")
    generator = torch.Generator().manual_seed(RANDOM_SEED)
    # random_split ожидает объект Dataset или последовательность индексов
    train_subset, val_subset = random_split(range(len(train_df_full)), [train_size, val_size], generator=generator)

    # --- ИСПРАВЛЕНИЕ ЗДЕСЬ ---
    # Используем .indices для получения списка индексов из Subset
    train_split_df = train_df_full.iloc[train_subset.indices].copy().reset_index(drop=True)
    val_split_df = train_df_full.iloc[val_subset.indices].copy().reset_index(drop=True)
    # -----------------------

    print(f"Данные успешно разделены: Train={len(train_split_df)}, Val={len(val_split_df)}")

except FileNotFoundError as e:
    print(f"❌ Ошибка: Не найден один из CSV файлов ({e}). Убедитесь, что файлы {TRAIN_CSV_FILENAME} и {TEST_CSV_FILENAME} находятся в {DATA_DIR.resolve()}")
    raise
except ValueError as e:
    print(f"❌ Ошибка при обработке данных: {e}")
    raise
except Exception as e:
    print(f"❌ Неизвестная ошибка при загрузке/подготовке данных: {e}")
    traceback.print_exc(limit=1)
    raise

# --- Проверка наличия необходимых переменных (теперь должна пройти) ---
required_vars = ['test_df', 'val_split_df', 'char_to_int', 'int_to_char']
if not all(var in locals() or var in globals() for var in required_vars):
    # Эта ошибка больше не должна возникать, но оставим проверку на всякий случай
    raise NameError("Критическая ошибка: Переменные test_df, val_split_df, char_to_int, int_to_char не были определены после загрузки!")
else:
    print("\nПеременные test_df, val_split_df, char_to_int, int_to_char успешно определены.")


# --- 4. Вспомогательные функции загрузки ---
def extract_lev_from_filename(filename: str) -> float:
    """Извлекает Levenshtein из имени файла формата ..._levX.XXXX.pth"""
    match = re.search(r"lev([\d.]+)\.pth$", filename)
    if match:
        try: return float(match.group(1))
        except ValueError: return float('inf')
    return float('inf')

def load_model_from_checkpoint(model_info: Dict[str, str], device: torch.device) -> Tuple[Optional[nn.Module], Optional[Dict]]:
    """Загружает модель и конфиг по информации из словаря model_info."""
    model_path = Path(model_info['model_path'])
    config_path = Path(model_info['config_path'])
    model_name = model_info['name']

    if not model_path.exists(): print(f"Ошибка: Файл модели не найден для '{model_name}': {model_path}"); return None, None
    if not config_path.exists(): print(f"Ошибка: Файл конфигурации не найден для '{model_name}': {config_path}"); return None, None

    print(f"\nЗагрузка модели '{model_name}'...")
    try:
        # Загрузка конфига
        with open(config_path, 'r') as f: config = json.load(f)
        # Убедимся, что vocab_size есть в конфиге модели (используем глобально созданный словарь)
        if 'model' in config and 'vocab_size' not in config['model']:
             config['model']['vocab_size'] = vocab_size # Используем vocab_size, полученный выше
             print(f"  Добавлен vocab_size={vocab_size} в загруженный конфиг '{model_name}'.")
        # Добавим глобальные параметры в конфиг для использования в Dataset
        if 'paths' not in config: config['paths'] = {}
        config['paths']['data_dir'] = str(DATA_DIR) # Dataset ожидает строку или Path
        config['morse_code_column'] = MORSE_CODE_COLUMN
        config['test_file_column'] = FILE_ID_COLUMN # Используем глобальную переменную
        config['train_file_column'] = FILE_ID_COLUMN # Добавим и для train на всякий случай
        if 'ctc' not in config: config['ctc'] = {}
        config['ctc']['pad_idx'] = PAD_IDX
        config['ctc']['blank_idx'] = BLANK_IDX

        # Инициализация модели
        model = MorseRecognizer(config).to(device)

        # Загрузка весов
        checkpoint = torch.load(model_path, map_location=device)
        state_dict_to_load = None
        if isinstance(checkpoint, dict):
            if 'model_state_dict' in checkpoint: state_dict_to_load = checkpoint['model_state_dict']
            elif 'state_dict' in checkpoint: state_dict_to_load = checkpoint['state_dict']
        if state_dict_to_load is None: state_dict_to_load = checkpoint
        new_state_dict = OrderedDict()
        has_module_prefix = any(k.startswith('module.') for k in state_dict_to_load.keys())
        for k, v in state_dict_to_load.items():
            name = k[7:] if has_module_prefix and k.startswith('module.') else k
            new_state_dict[name] = v
        model.load_state_dict(new_state_dict)
        model.eval() # Переводим в режим оценки
        print(f"  Модель '{model_name}' успешно загружена и переведена в режим eval().")
        return model, config
    except Exception as e:
        print(f"❌ Ошибка при загрузке модели '{model_name}': {e}")
        traceback.print_exc(limit=1)
        return None, None

# --- 5. Загрузка всех моделей ---
loaded_models_data = {}
print("\n" + "="*30 + " Загрузка Моделей " + "="*30)
for model_info in MODELS_TO_ANALYZE:
    model, config = load_model_from_checkpoint(model_info, DEVICE)
    if model and config:
        base_lev = extract_lev_from_filename(model_info['model_path'])
        loaded_models_data[model_info['name']] = {
            'model': model,
            'config': config, # Сохраняем модифицированный конфиг
            'base_lev': base_lev
        }
        print(f"  Levenshtein из имени файла для '{model_info['name']}': {base_lev:.4f}")

if not loaded_models_data:
    raise RuntimeError("Не удалось загрузить ни одной модели. Проверьте пути и ошибки выше.")
else:
    print(f"\nУспешно загружено моделей: {len(loaded_models_data)}")


--- Загрузка данных и создание словарей ---
Загружен train.csv: 30000 строк
Загружен sample_submission.csv: 5000 строк
Словарь создан: 45 символов (включая бланк).
  Blank: '<blank>' (0), Pad для collate: '<pad>' (-1)
Разделение 30000 записей на Train (27000) и Val (3000) с Seed: 42
Данные успешно разделены: Train=27000, Val=3000

Переменные test_df, val_split_df, char_to_int, int_to_char успешно определены.

============================== Загрузка Моделей ==============================

Загрузка модели '0.2637'...
  Модель '0.2637' успешно загружена и переведена в режим eval().
  Levenshtein из имени файла для '0.2637': 0.2637

Загрузка модели '0.2638'...
  Добавлен vocab_size=45 в загруженный конфиг '0.2638'.
  Модель '0.2638' успешно загружена и переведена в режим eval().
  Levenshtein из имени файла для '0.2638': 0.2637

Загрузка модели '0.2657'...
  Модель '0.2657' успешно загружена и переведена в режим eval().
  Levenshtein из имени файла для '0.2657': 0.2657

Загрузка модели '0

### Ячейка 4: Функции Инференса и Анализа Ошибок

In [15]:
### Ячейка 4: Функции Инференса и Анализа Ошибок
# Ячейка 4: Функции Инференса и Анализа Ошибок (v1.1 - Исправлена логика analyze_character_errors)

import traceback # Добавим на всякий случай
from collections import defaultdict, Counter
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import Levenshtein
from typing import List, Dict, Tuple, Optional, Any
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np # Нужен для np.isfinite
import gc

# --- Убедимся, что нужные классы/функции из Ячейки 2 доступны ---
# MorseRecognizer, MorseDataset, collate_fn, ctc_greedy_decode

def get_predictions(
    model_name: str,
    loaded_models_data: Dict[str, Dict],
    df: pd.DataFrame, # Либо val_split_df, либо test_df
    char_to_int: Dict[str, int],
    int_to_char: Dict[int, str],
    device: torch.device,
    is_validation: bool # Флаг для определения, нужны ли реальные тексты
) -> Tuple[List[str], Optional[List[str]]]:
    """Запускает инференс для указанной модели на данных из df."""
    if model_name not in loaded_models_data:
        print(f"Ошибка: Модель '{model_name}' не найдена в загруженных данных.")
        return [], None

    model_data = loaded_models_data[model_name]
    model = model_data['model']
    config = model_data['config']
    base_lev = model_data['base_lev']
    dataset_type = "Validation" if is_validation else "Test"
    print(f"\n--- Запуск инференса ({dataset_type}) для модели: '{model_name}' (Base Lev: {base_lev:.4f}) ---")

    # Получаем параметры из специфичного конфига модели
    pad_idx = config.get("ctc", {}).get("pad_idx", -1)
    blank_idx = config.get("ctc", {}).get("blank_idx", 0)
    text_col = config.get('morse_code_column', 'message')
    batch_size_key = "batch_size" # Ищем ключ батчсайза
    inference_batch_size = config.get("finetuning", {}).get(batch_size_key, config.get("training", {}).get(batch_size_key, 16)) * 2
    num_workers = config.get("num_workers", 0)

    # Создаем Dataset и DataLoader с конфигом модели
    try:
        # Передаем text_col в конструктор Dataset
        dataset = MorseDataset(df=df, char_to_int=char_to_int, config=config, is_train=False, text_column=text_col)
        collate_wrapper = lambda batch: collate_fn(batch, pad_idx)
        dataloader = DataLoader(dataset, batch_size=inference_batch_size, shuffle=False,
                                num_workers=num_workers, collate_fn=collate_wrapper, pin_memory=True)
    except Exception as e:
        print(f"❌ Ошибка создания DataLoader для '{model_name}': {e}")
        traceback.print_exc(limit=1) # Печатаем traceback для диагностики
        return [], None

    all_predictions = []
    all_ground_truth = [] if is_validation else None

    pbar = tqdm(dataloader, desc=f"Инференс '{model_name}'", leave=False)
    with torch.no_grad():
        for batch_idx, batch_data in enumerate(pbar): # Добавим enumerate для отладки батчей
            if batch_data is None:
                print(f"Предупреждение (Инференс {model_name}, Батч {batch_idx}): Пропущен None батч.")
                continue
            try:
                features, targets, _, target_lengths = batch_data # targets и target_lengths нужны для валидации
                features = features.to(device)
            except Exception as e_batch:
                print(f"\nОшибка обработки батча {batch_idx} для '{model_name}': {e_batch}")
                # Добавляем ошибки, чтобы сохранить соответствие длины
                batch_size_est = batch_data[0].shape[0] if batch_data and batch_data[0] is not None else inference_batch_size
                all_predictions.extend(["ERROR_BATCH"] * batch_size_est)
                if is_validation: all_ground_truth.extend(["ERROR_BATCH_GT"] * batch_size_est)
                continue

            try:
                logits = model(features)
                # Проверка на NaN/Inf в логитах перед декодированием
                if not torch.isfinite(logits).all():
                    print(f"\nПредупреждение (Инференс {model_name}, Батч {batch_idx}): Обнаружены NaN/Inf в логитах! Заменяем на пустые предсказания.")
                    decoded_batch = ["ERROR_LOGITS"] * features.shape[0]
                else:
                    decoded_batch = ctc_greedy_decode(logits.detach(), int_to_char, blank_idx)

                all_predictions.extend(decoded_batch)

                # Собираем реальные тексты, если это валидация
                if is_validation:
                    targets_cpu = targets.cpu()
                    target_lengths_cpu = target_lengths.cpu().numpy()
                    for i in range(targets_cpu.size(0)):
                        length = target_lengths_cpu[i]
                        if length > 0 and length <= targets_cpu.shape[1]: # Добавлена проверка границы
                            target_indices = targets_cpu[i, :length].numpy()
                            gt_text = "".join([int_to_char.get(idx, '?') for idx in target_indices if idx != pad_idx and idx != blank_idx])
                            all_ground_truth.append(gt_text)
                        elif length > targets_cpu.shape[1]:
                             print(f"Предупреждение (Инференс {model_name}, Батч {batch_idx}): Некорректная длина цели {length} > {targets_cpu.shape[1]}.")
                             all_ground_truth.append("ERROR_GT_LEN")
                        else: # length == 0
                            all_ground_truth.append("") # Пустая строка для нулевой длины

            except RuntimeError as e:
                 if "CUDA out of memory" in str(e):
                     print(f"\n❌ OOM ОШИБКА (Инференс {model_name}, Батч {batch_idx})! Заменяем на ошибки.")
                     decoded_batch = ["ERROR_OOM"] * features.shape[0]
                     all_predictions.extend(decoded_batch)
                     if is_validation: all_ground_truth.extend(["ERROR_OOM_GT"] * features.shape[0])
                     gc.collect(); torch.cuda.empty_cache()
                 else:
                     print(f"\n❌ RuntimeError (Инференс {model_name}, Батч {batch_idx}): {e}")
                     decoded_batch = ["ERROR_RUNTIME"] * features.shape[0]
                     all_predictions.extend(decoded_batch)
                     if is_validation: all_ground_truth.extend(["ERROR_RUNTIME_GT"] * features.shape[0])
            except Exception as e_infer:
                print(f"\n❌ Ошибка инференса/декодирования (Батч {batch_idx}) для '{model_name}': {e_infer}")
                batch_size_est = features.shape[0]
                all_predictions.extend(["ERROR_INFER"] * batch_size_est)
                if is_validation: all_ground_truth.extend(["ERROR_INFER_GT"] * batch_size_est)

            del features, logits, batch_data, targets, target_lengths # Очистка

    print(f"--- Инференс ({dataset_type}) для '{model_name}' завершен. Предсказаний: {len(all_predictions)} ---")
    # Дополнительная проверка совпадения длин на выходе
    if is_validation and len(all_predictions) != len(all_ground_truth):
         print(f"!!! КРИТИЧЕСКОЕ ПРЕДУПРЕЖДЕНИЕ ({model_name}, {dataset_type}): Несовпадение длин предсказаний ({len(all_predictions)}) и GT ({len(all_ground_truth)})!")
         # Пытаемся выровнять (хотя это плохо)
         min_len = min(len(all_predictions), len(all_ground_truth))
         all_predictions = all_predictions[:min_len]
         all_ground_truth = all_ground_truth[:min_len]

    return all_predictions, all_ground_truth


def analyze_character_errors(predictions: List[str], ground_truth: List[str]) -> Dict[str, Any]:
    """Анализирует ошибки посимвольно (v1.1 - Исправлена логика)."""
    if len(predictions) != len(ground_truth):
        print(f"Ошибка анализа: количество предсказаний ({len(predictions)}) не совпадает с количеством реальных строк ({len(ground_truth)})!")
        # Возвращаем пустой словарь или генерируем исключение
        return {
            'substitutions': {}, 'insertions': {}, 'deletions': {},
            'total_errors': -1, 'total_chars_gt': -1, 'character_error_rate': -1.0
        }

    substitutions = defaultdict(lambda: defaultdict(int))
    insertions = defaultdict(int)
    deletions = defaultdict(int)
    total_errors = 0
    total_chars_gt = 0
    num_pairs_processed = 0
    num_lev_errors = 0

    print(f"Анализ ошибок: Обработка {len(predictions)} пар строк...")
    pbar_analyze = tqdm(zip(predictions, ground_truth), total=len(predictions), desc="Анализ ошибок", leave=False)

    for i, (pred, gt) in enumerate(pbar_analyze):
        # Проверка типов перед обработкой
        if not isinstance(pred, str) or not isinstance(gt, str):
            # print(f"Предупреждение (Анализ, пара {i}): Некорректные типы строк (pred: {type(pred)}, gt: {type(gt)}). Пропуск.")
            num_lev_errors += 1
            continue

        total_chars_gt += len(gt)
        try:
            # editops(s1, s2) -> операции для преобразования s1 в s2
            # s1 = pred, s2 = gt
            # op = (tag, pos_s1, pos_s2)
            ops = Levenshtein.editops(pred, gt)
            total_errors += len(ops)
            num_pairs_processed += 1

            for op_type, pos_pred, pos_gt in ops:
                # --- ИСПРАВЛЕННАЯ ЛОГИКА ---
                if op_type == 'replace': # Замена: pred[pos_pred] заменяется на gt[pos_gt]
                    # Проверка индексов перед доступом
                    if 0 <= pos_gt < len(gt) and 0 <= pos_pred < len(pred):
                        substitutions[gt[pos_gt]][pred[pos_pred]] += 1
                    else:
                        # print(f"Предупреждение (replace, пара {i}): Индекс вне диапазона (pos_gt={pos_gt}, len_gt={len(gt)}; pos_pred={pos_pred}, len_pred={len(pred)}).")
                        num_lev_errors += 1 # Считаем как ошибку обработки
                elif op_type == 'insert': # Вставка: gt[pos_gt] вставляется перед pred[pos_pred]
                    # Проверка индекса перед доступом
                    if 0 <= pos_gt < len(gt):
                        insertions[gt[pos_gt]] += 1 # Записываем символ из GT, который был вставлен
                    else:
                        # print(f"Предупреждение (insert, пара {i}): Индекс gt вне диапазона (pos_gt={pos_gt}, len_gt={len(gt)}).")
                        num_lev_errors += 1
                elif op_type == 'delete': # Удаление: pred[pos_pred] удаляется
                     # Проверка индекса перед доступом
                    if 0 <= pos_pred < len(pred):
                        deletions[pred[pos_pred]] += 1 # Записываем символ из PRED, который был удален
                    else:
                        # print(f"Предупреждение (delete, пара {i}): Индекс pred вне диапазона (pos_pred={pos_pred}, len_pred={len(pred)}).")
                        num_lev_errors += 1
                # ---------------------------

        except Exception as e:
            # Убрал печать ошибки для каждой строки, чтобы не засорять вывод
            # print(f"Ошибка при вычислении editops для: P='{pred}', GT='{gt}'. Ошибка: {e}")
            num_lev_errors += 1

    pbar_analyze.close()
    if num_lev_errors > 0:
         print(f"Предупреждение: Обнаружено {num_lev_errors} ошибок при вычислении/обработке Levenshtein editops.")

    # Преобразование defaultdict в обычные dict для вывода/сохранения
    results = {
        'substitutions': {k: dict(v) for k, v in substitutions.items()},
        'insertions': dict(insertions),
        'deletions': dict(deletions),
        'total_errors': total_errors,
        'total_chars_gt': total_chars_gt,
        'character_error_rate': (total_errors / total_chars_gt) if total_chars_gt > 0 else 0.0,
        'pairs_processed_successfully': num_pairs_processed,
        'levenshtein_processing_errors': num_lev_errors
    }
    print(f"Анализ завершен. Успешно обработано пар: {num_pairs_processed}. Ошибок обработки Levenshtein: {num_lev_errors}.")
    return results


def print_top_errors(error_stats: Dict[str, Any], top_n: int = 10):
    """Выводит топ N ошибок из статистики."""
    print(f"\n  Общая ошибка по символам (CER): {error_stats.get('character_error_rate', 0.0):.4f}")

    # Замены
    subs_flat = []
    for gt_char, pred_map in error_stats.get('substitutions', {}).items():
        for pred_char, count in pred_map.items():
            subs_flat.append(((gt_char, pred_char), count))
    subs_flat.sort(key=lambda item: item[1], reverse=True)
    print(f"\n  Топ-{top_n} замен (Реальный -> Предсказанный):")
    if not subs_flat: print("    Нет замен.")
    for (gt, pred), count in subs_flat[:top_n]:
        print(f"    '{gt}' -> '{pred}': {count} раз")

    # Вставки (Символы из GT, которые были вставлены, чтобы получить GT из Pred)
    # То есть, это символы, которые были ПРОПУЩЕНЫ моделью.
    ins_list = sorted(error_stats.get('insertions', {}).items(), key=lambda item: item[1], reverse=True)
    print(f"\n  Топ-{top_n} вставок (Пропущенные символы из GT):")
    if not ins_list: print("    Нет вставок.")
    for char, count in ins_list[:top_n]:
        print(f"    '{char}': {count} раз")

    # Удаления (Символы из Pred, которые были удалены, чтобы получить GT из Pred)
    # То есть, это символы, которые были ЛИШНИМИ в предсказании модели.
    del_list = sorted(error_stats.get('deletions', {}).items(), key=lambda item: item[1], reverse=True)
    print(f"\n  Топ-{top_n} удалений (Лишние символы в Pred):")
    if not del_list: print("    Нет удалений.")
    for char, count in del_list[:top_n]:
        print(f"    '{char}': {count} раз")

print("Функции инференса и анализа ошибок определены (v1.1).")

Функции инференса и анализа ошибок определены (v1.1).


### Ячейка 5: Запуск Анализа Ошибок на Валидации

In [16]:
# Ячейка 5: Запуск Анализа Ошибок на Валидации

all_error_stats = {}

print("\n" + "="*30 + " Анализ Ошибок Моделей на Валидации " + "="*30)

for model_name in loaded_models_data.keys():
    # Получаем предсказания и реальные тексты
    val_preds, val_gt = get_predictions(
        model_name=model_name,
        loaded_models_data=loaded_models_data,
        df=val_split_df, # Используем валидационный DataFrame
        char_to_int=char_to_int,
        int_to_char=int_to_char,
        device=DEVICE,
        is_validation=True # Указываем, что это валидация
    )

    if not val_preds or not val_gt or len(val_preds) != len(val_gt):
        print(f"Пропуск анализа ошибок для '{model_name}' из-за проблем с инференсом.")
        continue

    # Анализируем ошибки
    print(f"\nАнализ ошибок для модели: '{model_name}'...")
    error_stats = analyze_character_errors(val_preds, val_gt)
    all_error_stats[model_name] = error_stats

    # Выводим топ ошибок
    print_top_errors(error_stats, top_n=10)

    # Опционально: Простая визуализация общего количества ошибок по символам
    all_char_errors = defaultdict(int)
    for gt_char, pred_map in error_stats.get('substitutions', {}).items():
        all_char_errors[gt_char] += sum(pred_map.values()) # Ошибки, когда gt_char был заменен
    for ins_char, count in error_stats.get('insertions', {}).items():
        all_char_errors[ins_char] += count # Ошибки, когда ins_char был вставлен
    for del_char, count in error_stats.get('deletions', {}).items():
        all_char_errors[del_char] += count # Ошибки, когда del_char был удален

    if all_char_errors:
        sorted_char_errors = sorted(all_char_errors.items(), key=lambda item: item[1], reverse=True)
        chars, counts = zip(*sorted_char_errors[:20]) # Берем топ-20 для графика

        plt.figure(figsize=(12, 6))
        sns.barplot(x=list(chars), y=list(counts), palette="viridis")
        plt.title(f"Топ-20 символов по общему числу ошибок ({model_name})")
        plt.xlabel("Символ")
        plt.ylabel("Количество ошибок (Замены+Вставки+Удаления)")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

    # Очистка памяти
    del val_preds, val_gt, error_stats
    gc.collect()
    if DEVICE == torch.device('cuda'): torch.cuda.empty_cache()

print("\nАнализ ошибок на валидации завершен.")


============================== Анализ Ошибок Моделей на Валидации ==============================

--- Запуск инференса (Validation) для модели: '0.2637' (Base Lev: 0.2637) ---


Инференс '0.2637':   0%|          | 0/94 [00:00<?, ?it/s]

KeyboardInterrupt: 

### Ячейка 6: Генерация Предсказаний на Тестовом Наборе

In [ ]:
# Ячейка 6: Генерация Предсказаний на Тестовом Наборе

all_test_predictions = {}

print("\n" + "="*30 + " Генерация Предсказаний на Тестовом Наборе " + "="*30)

for model_name in loaded_models_data.keys():
    # Получаем предсказания для тестового набора
    test_preds, _ = get_predictions(
        model_name=model_name,
        loaded_models_data=loaded_models_data,
        df=test_df, # Используем тестовый DataFrame
        char_to_int=char_to_int,
        int_to_char=int_to_char,
        device=DEVICE,
        is_validation=False # Указываем, что это НЕ валидация
    )

    if not test_preds or len(test_preds) != len(test_df):
        print(f"⚠️ Предупреждение: Количество предсказаний ({len(test_preds)}) для '{model_name}' не совпадает с размером test_df ({len(test_df)}). Пропуск.")
        # Можно добавить обработку ошибок, например, заполнить недостающие предсказания
    else:
        all_test_predictions[model_name] = test_preds
        print(f"Предсказания для '{model_name}' на тесте получены.")

    # Очистка памяти
    del test_preds
    gc.collect()
    if DEVICE == torch.device('cuda'): torch.cuda.empty_cache()

if len(all_test_predictions) != len(loaded_models_data):
    print("\n⚠️ Предупреждение: Не для всех моделей удалось сгенерировать предсказания на тесте!")
elif not all_test_predictions:
     raise RuntimeError("Не удалось сгенерировать предсказания ни для одной модели!")
else:
    print(f"\nПредсказания на тесте сгенерированы для {len(all_test_predictions)} моделей.")


============================== Генерация Предсказаний на Тестовом Наборе ==============================

--- Запуск инференса (Test) для модели: '0.2637' (Base Lev: 0.2637) ---


Инференс '0.2637':   0%|          | 0/157 [00:00<?, ?it/s]

--- Инференс (Test) для '0.2637' завершен. Предсказаний: 5000 ---
Предсказания для '0.2637' на тесте получены.

--- Запуск инференса (Test) для модели: '0.2638' (Base Lev: 0.2637) ---


Инференс '0.2638':   0%|          | 0/313 [00:00<?, ?it/s]

--- Инференс (Test) для '0.2638' завершен. Предсказаний: 5000 ---
Предсказания для '0.2638' на тесте получены.

--- Запуск инференса (Test) для модели: '0.2657' (Base Lev: 0.2657) ---


...

--- Инференс (Test) для '0.2773' завершен. Предсказаний: 5000 ---
Предсказания для '0.2773' на тесте получены.

--- Запуск инференса (Test) для модели: '0.3133' (Base Lev: 0.3133) ---


Инференс '0.3133':   0%|          | 0/313 [00:00<?, ?it/s]

--- Инференс (Test) для '0.3133' завершен. Предсказаний: 5000 ---
Предсказания для '0.3133' на тесте получены.

--- Запуск инференса (Test) для модели: '0.2607' (Base Lev: 0.2607) ---


Инференс '0.2607':   0%|          | 0/157 [00:00<?, ?it/s]

--- Инференс (Test) для '0.2607' завершен. Предсказаний: 5000 ---
Предсказания для '0.2607' на тесте получены.

Предсказания на тесте сгенерированы для 7 моделей.


### Ячейка 7: Ансамблирование (Простое Голосование)

In [ ]:
### Ячейка 7: Ансамблирование (Простое Голосование)
# Ячейка 7: Ансамблирование (Простое Голосование) (v1.2 - Исправлен NameError для ensemble_df)

from collections import Counter # Убедимся, что Counter импортирован
import pandas as pd
from tqdm.notebook import tqdm
from typing import Dict, List
import traceback # Добавим для отладки

def ensemble_majority_vote(
    all_test_predictions: Dict[str, List[str]],
    test_ids: List[str],
    id_column: str, # Принимаем имя колонки как аргумент
    pred_column: str # Принимаем имя колонки как аргумент
) -> pd.DataFrame:
    """Выполняет ансамблирование методом простого большинства голосов."""
    print("\n--- Запуск ансамблирования (Majority Vote) ---")
    if not all_test_predictions:
        print("Ошибка: Нет предсказаний для ансамблирования.")
        return pd.DataFrame(columns=[id_column, pred_column])

    model_names = list(all_test_predictions.keys())
    num_models = len(model_names)
    num_predictions = len(test_ids)

    # Проверка консистентности количества предсказаний
    for name in model_names:
        # Добавим проверку на случай, если предсказания для модели None или не список
        if not isinstance(all_test_predictions.get(name), list) or len(all_test_predictions[name]) != num_predictions:
            raise ValueError(f"Количество или тип предсказаний для модели '{name}' ({len(all_test_predictions.get(name, []))}) некорректны или не совпадают с количеством тестовых ID ({num_predictions})!")

    final_predictions = []
    pbar = tqdm(range(num_predictions), desc="Ансамблирование", leave=False)
    for i in pbar:
        current_preds = [all_test_predictions[name][i] for name in model_names if i < len(all_test_predictions[name])] # Добавлена проверка индекса
        # Используем Counter для подсчета голосов
        # Исключаем возможные None или не-строковые значения перед голосованием
        valid_preds = [p for p in current_preds if isinstance(p, str)]
        if not valid_preds:
             # Если нет валидных предсказаний для этого сэмпла, берем от первой модели или ставим ошибку
             final_predictions.append(current_preds[0] if current_preds else "ERROR_NO_VALID_VOTES")
             continue

        vote_counts = Counter(valid_preds)
        # Находим самый частый элемент (или один из них, если есть ничья)
        most_common = vote_counts.most_common(1)
        if most_common:
            final_predictions.append(most_common[0][0])
        else:
            # Если вдруг valid_preds был пуст (не должно случиться при проверке выше)
            final_predictions.append("ERROR_NO_VOTES") # Или использовать предсказание первой модели

    # Создаем финальный DataFrame
    ensemble_df = pd.DataFrame({
        id_column: test_ids,
        pred_column: final_predictions
    })
    print("--- Ансамблирование завершено ---")
    return ensemble_df

# --- Запуск ансамблирования ---
ensemble_df = None # Инициализируем ensemble_df как None по умолчанию
test_id_col = None # Инициализируем имена колонок
pred_col = None

if 'all_test_predictions' in locals() and all_test_predictions:
    # Получаем имена колонок из глобальной области видимости (определены в Ячейке 3)
    try:
        test_id_col = FILE_ID_COLUMN
        pred_col = MORSE_CODE_COLUMN
        print(f"Используются колонки: ID='{test_id_col}', Pred='{pred_col}'")
    except NameError:
        print("Критическая ошибка: Переменные FILE_ID_COLUMN или MORSE_CODE_COLUMN не определены.")
        print("Убедитесь, что Ячейка 3 была успешно выполнена перед Ячейкой 7.")
        # test_id_col и pred_col останутся None

    # --- Основная логика ансамблирования запускается только если колонки найдены ---
    if test_id_col is not None and pred_col is not None:
        try:
            # Убедимся, что test_df существует
            if 'test_df' not in locals():
                raise NameError("Переменная test_df не определена.")

            test_ids_list = test_df[test_id_col].tolist()
            # Вызываем функцию ансамблирования
            ensemble_df = ensemble_majority_vote(
                all_test_predictions,
                test_ids_list,
                test_id_col, # Передаем имя колонки
                pred_col   # Передаем имя колонки
            )
            # Проверяем результат ансамблирования
            if ensemble_df is not None and not ensemble_df.empty:
                 print(f"Финальный DataFrame ансамбля создан. Размер: {ensemble_df.shape}")
                 print("Примеры ансамблированных предсказаний:")
                 print(ensemble_df.head())
            else:
                 print("Функция ансамблирования вернула пустой или некорректный результат.")
                 ensemble_df = None # Убедимся, что None в случае неудачи

        except NameError as ne:
             print(f"Ошибка NameError при подготовке к ансамблированию: {ne}")
             ensemble_df = None
        except KeyError as ke:
             print(f"Ошибка KeyError: Колонка '{ke}' не найдена в test_df.")
             ensemble_df = None
        except Exception as e:
             print(f"Неизвестная ошибка при запуске ансамблирования: {e}")
             traceback.print_exc(limit=1) # Печатаем traceback для неизвестных ошибок
             ensemble_df = None
    else:
         print("Ансамблирование не может быть выполнено, так как имена колонок не определены.")
         # ensemble_df остается None

else:
    print("Ансамблирование пропущено, так как нет валидных предсказаний от моделей ('all_test_predictions').")
    # ensemble_df уже None

# В конце этой ячейки, ensemble_df будет либо DataFrame с результатами, либо None.
# Ячейка 8 сможет безопасно проверить `if ensemble_df is not None:`

Используются колонки: ID='id', Pred='message'

--- Запуск ансамблирования (Majority Vote) ---


Ансамблирование:   0%|          | 0/5000 [00:00<?, ?it/s]

--- Ансамблирование завершено ---
Финальный DataFrame ансамбля создан. Размер: (5000, 2)
Примеры ансамблированных предсказаний:
           id     message
0  30001.opus   ЯЮ6ЛИТЖБШ
1  30002.opus    КЩ В9Ю 9
2  30003.opus     Ы65Ф61Я
3  30004.opus  ЖЖНЖ9РЫНЦ3
4  30005.opus     ЕЯФ4ЮЧЬ


### Ячейка 8: Сохранение Финального Submission

In [ ]:
# Ячейка 8: Сохранение Финального Submission

# --- Определяем имя файла ---
output_dir = Path("./ensemble_output") # Создадим отдельную папку для результатов ансамбля
output_dir.mkdir(parents=True, exist_ok=True)
submission_filename = output_dir / "submission_ensemble_majority.csv"

# --- Сохранение ---
if ensemble_df is not None:
    try:
        ensemble_df.to_csv(submission_filename, index=False)
        print(f"\n✅ Финальный ансамблированный submission сохранен в: {submission_filename}")
    except Exception as e:
        print(f"\n❌ Ошибка при сохранении финального submission: {e}")
else:
    print("\nФинальный submission не был создан, так как ансамблирование не выполнялось.")

print("\nРабота ноутбука завершена.")


✅ Финальный ансамблированный submission сохранен в: ensemble_output\submission_ensemble_majority.csv

Работа ноутбука завершена.


In [ ]:
### Ячейка 9: Извлечение Послания из Последних 17 Файлов
import pandas as pd

print("\n" + "="*30 + " Извлечение Послания " + "="*30)

N_LAST_FILES = 17
ARTIFACT_CHARS_TO_REMOVE_SUFFIX = '0Ъ' # Символы для удаления с КОНЦА каждой части предсказания
# Символ '#' пока оставляем, т.к. его роль неясна (может быть частью сообщения)

message_source_name = None
source_predictions_df = None

# --- Определяем источник предсказаний ---
# Приоритет: Ансамбль, затем первая модель из загруженных
if 'ensemble_df' in locals() and ensemble_df is not None and not ensemble_df.empty:
    message_source_name = "Ансамбль (Majority Vote)"
    source_predictions_df = ensemble_df
    print(f"Используются предсказания из: {message_source_name}")
elif 'all_test_predictions' in locals() and all_test_predictions:
    # Берем предсказания первой модели из словаря как запасной вариант
    first_model_name = list(all_test_predictions.keys())[0]
    message_source_name = f"Одиночная модель: {first_model_name}"
    # Создаем DataFrame из предсказаний этой модели
    try:
        test_id_col = FILE_ID_COLUMN
        pred_col = MORSE_CODE_COLUMN
        test_ids_list = test_df[test_id_col].tolist()
        preds_list = all_test_predictions[first_model_name]
        if len(test_ids_list) == len(preds_list):
            source_predictions_df = pd.DataFrame({
                test_id_col: test_ids_list,
                pred_col: preds_list
            })
            print(f"Используются предсказания из: {message_source_name}")
        else:
            print(f"Ошибка: Несовпадение длин ID ({len(test_ids_list)}) и предсказаний ({len(preds_list)}) для модели {first_model_name}.")
            message_source_name = None # Сбрасываем имя источника
    except NameError:
         print("Ошибка: Переменные FILE_ID_COLUMN, MORSE_CODE_COLUMN или test_df не определены.")
         message_source_name = None
    except KeyError as e:
         print(f"Ошибка: Колонка '{e}' не найдена в test_df.")
         message_source_name = None
else:
    print("Ошибка: Нет доступных предсказаний (ни ансамбля, ни одиночных моделей) для извлечения послания.")

# --- Извлечение и обработка послания ---
final_message = None
if source_predictions_df is not None:
    try:
        # Получаем имена колонок (должны быть определены в Ячейке 3)
        id_col = FILE_ID_COLUMN
        pred_col = MORSE_CODE_COLUMN

        # Получаем ID файлов в оригинальном порядке из test_df
        all_test_ids = test_df[id_col].tolist()

        if len(all_test_ids) < N_LAST_FILES:
            print(f"Ошибка: В тестовом наборе ({len(all_test_ids)}) меньше файлов, чем требуется ({N_LAST_FILES}).")
        else:
            # Выбираем ID последних N файлов
            last_n_ids = all_test_ids[-N_LAST_FILES:]
            print(f"ID последних {N_LAST_FILES} файлов: {last_n_ids}")

            # Создаем словарь для быстрого поиска предсказаний по ID
            id_to_prediction_map = pd.Series(
                source_predictions_df[pred_col].values,
                index=source_predictions_df[id_col]
            ).to_dict()

            # Собираем предсказания для последних N файлов в правильном порядке
            message_parts = []
            missing_ids = []
            for file_id in last_n_ids:
                prediction = id_to_prediction_map.get(file_id)
                if prediction is not None and isinstance(prediction, str):
                    # Очищаем артефакты с конца КАЖДОГО предсказания перед склейкой
                    cleaned_part = prediction.rstrip(ARTIFACT_CHARS_TO_REMOVE_SUFFIX)
                    message_parts.append(cleaned_part)
                elif prediction is None:
                    missing_ids.append(file_id)
                    message_parts.append(f"[MISSING_PRED_FOR_{file_id}]") # Добавляем маркер ошибки
                else: # Если предсказание не строка (например, ошибка)
                     message_parts.append(f"[{str(prediction)}]") # Добавляем маркер ошибки

            if missing_ids:
                 print(f"Предупреждение: Не найдены предсказания для следующих ID: {missing_ids}")

            # Склеиваем очищенные части
            final_message = "".join(message_parts)

    except NameError:
        print("Критическая ошибка: Переменные FILE_ID_COLUMN, MORSE_CODE_COLUMN или test_df не определены.")
        final_message = "[ОШИБКА: НЕОБХОДИМЫЕ ПЕРЕМЕННЫЕ НЕ ОПРЕДЕЛЕНЫ]"
    except KeyError as e:
        print(f"Критическая ошибка: Колонка '{e}' не найдена в DataFrame.")
        final_message = f"[ОШИБКА: КОЛОНКА {e} НЕ НАЙДЕНА]"
    except Exception as e:
        print(f"Неизвестная ошибка при извлечении послания: {e}")
        final_message = f"[ОШИБКА: {type(e).__name__}]"

# --- Вывод результата ---
if final_message is not None:
    print("\n" + "-"*15 + " Расшифрованное Послание " + "-"*15)
    print(f"(Источник: {message_source_name})")
    print("\n" + final_message)
    print("\n" + "-" * (32 + len(" Расшифрованное Послание ")))
    print(f"\nПримечание: Из предсказаний удалены только символы '{ARTIFACT_CHARS_TO_REMOVE_SUFFIX}' с конца каждой из {N_LAST_FILES} частей.")
else:
    print("\nНе удалось извлечь послание из-за ошибок выше.")


============================== Извлечение Послания ==============================
Используются предсказания из: Ансамбль (Majority Vote)
ID последних 17 файлов: ['34984.opus', '34985.opus', '34986.opus', '34987.opus', '34988.opus', '34989.opus', '34990.opus', '34991.opus', '34992.opus', '34993.opus', '34994.opus', '34995.opus', '34996.opus', '34997.opus', '34998.opus', '34999.opus', '35000.opus']

--------------- Расшифрованное Послание ---------------
(Источник: Ансамбль (Majority Vote))

ДАМИНАМТОТИРСЫСАМЦИИЛ ЬСВЕДТКЧВНТИ ЯМДМЫМЮНЯМЦ ЮТИЫМ ЬТКТЧМЫН ДТЫМРГЗ ЕПИГАНХ ВСОЕГЬ Р ВКТДАМИ НКШМДНИ ГЕКНЖТАИЛ ДАСДП ЬСОЕМУНТИ ИГВКСОЕП ЬКСХЫСУС ХНУ ЮН ХНУСИЕТЬТКП ДСЯНКМЫОЦ ИМК #КНОЬКМ СОЕНЫМОП ЬСЮНВМ ИЛ ЬСОЕНДМЫМ ЙЫНУС ЙЫМЧАТУС ЬКТДЛХТ ЫМЖАЛШ ДЛУСВ М ЧТЫНАМБИЛ ДАСДП ОСЮВНЫМ НЮЙГРГТЫТУКНЩНАС АТ ИСЧТ КГЖНЕПОЦ ЮН НЙОСЫЗЕАЗ ДТКАСОЕПШМИДСЫСДТОЫЗДЛ ОЫЛ5МЕТ #ЕС ЬЫНАТ ЕСЗЕРЫ9АМЕ0ПАН ЖНОЕСЕТ ЬТКТВНЖМ ГДТЫМЖТААСБ АН 75 РМЫСЯМРЫСДОДЦЮП ЬСВВТКЧМДНТЕОЦ ДКГЖАГЗ АТГЕСИМИЛИМ ЬНЫПЯНИМ АНХМШ СЬТКНЕСКСДОРСКС ИЛ ДА

In [1]:
import pandas as pd
from collections import Counter
import numpy as np
import math
import re # На всякий случай

# Русский алфавит (буква ё обычно исключается или заменяется на е)
RUSSIAN_ALPHABET = 'АБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ'
ALPHABET_LEN = len(RUSSIAN_ALPHABET)

# Примерные частоты букв русского языка (в процентах, округленно)
# Источник: может варьироваться, взят для примера
RUSSIAN_FREQ = {
    'О': 11.0, 'Е': 8.5, 'А': 8.0, 'И': 7.4, 'Н': 6.7, 'Т': 6.2,
    'С': 5.5, 'Р': 4.7, 'В': 4.5, 'Л': 4.4, 'К': 3.5, 'М': 3.2,
    'Д': 3.0, 'П': 2.8, 'У': 2.6, 'Я': 2.0, 'Ы': 1.9, 'З': 1.8,
    'Ь': 1.7, 'Б': 1.6, 'Г': 1.6, 'Ч': 1.4, 'Й': 1.2, 'Х': 1.0,
    'Ж': 0.9, 'Ш': 0.7, 'Ю': 0.6, 'Ц': 0.5, 'Щ': 0.4, 'Э': 0.3,
    'Ф': 0.2, 'Ъ': 0.04
}
# Нормализуем частоты, чтобы сумма была близка к 1
total_freq = sum(RUSSIAN_FREQ.values())
RUSSIAN_FREQ_NORMALIZED = {char: freq / total_freq for char, freq in RUSSIAN_FREQ.items()}

# Консенсусный текст (без цифр и #)
ciphertext = "ДАМИНАМТОТИРСЫСАМЦИИЛ ЬСВЕДТКЧВНТИ ЯМДМЫМЮНЯМЦ ЮТИЫМ ЬТКТЧМЫН ДТЫМРГЗ ЕПИГАНХ ВСОЕГЬ Р ВКТДАМИ НКШМДНИ ГЕКНЖТАИЛ ДАСДП ЬСОЕМУНТИ ИГВКСОЕП ЬКСХЫСУС ХНУ ЮН ХНУСИЕТЬТКП ДСЯНКМЫОЦ ИМК КНОЬКМ СОЕНЫМОП ЬСЮНВМ ИЛ ЬСОЕНДМЫМ ЙЫНУС ЙЫМЧАТУС ЬКТДЛХТ ЫМЖАЛШ ДЛУСВ М ЧТЫНАМБИЛ ДАСДП ОСЮВНЫМ НЮЙГРГТЫТУКНЩНАС АТ ИСЧТ КГЖНЕПОЦ ЮН НЙОСЫЗЕАЗ ДТКАСОЕПШМИДСЫСДТОЫЗДЛ ОЫЛЕТ ЕС ЫНАТ ЕСЗЕРЫАМЕАН ЖНОЕСЕТ ЬТКТВНЖМ ГДТЫМЖТААСБ АН РМЫСЯМРЫСДОДЦЮП ЬСВВТКЧМДНТЕОЦ ДКГЖАГЗ АТГЕСИМИЛИМ ЬНЫПЯНИМ АНХМШ СЬТКНЕСКСДОРСКС ИЛ ДАСДП ЬСОЕМУАТИЕНБАЛ КНРТЕАЛШ ВДМУНЕТЫТБОИСЧТИ СЕЬКНДМЕП ЮН ДНИМ ОЬНОМЕТЫПАЛБ ЖТЫАСР ТОЫМ ДЛ ЬСЧТЫНТЕТ ДТКАГЕПОЦ ВСИСБ НН ОДТЕ ОСЫАЯН ВНКМЕ ЧМЮАП РНЧВСИГРСАТЯ ЬТКТВНЖМ"

# Удалим пробелы для чистоты анализа Виженера (они обычно не шифруются или шифруются отдельно)
ciphertext_no_spaces = "".join(ciphertext.split())
print(f"Длина текста без пробелов: {len(ciphertext_no_spaces)}")

# Предполагаемая длина ключа
KEY_LENGTH = 11

Длина текста без пробелов: 580


In [2]:
# Разделение на подтексты (колонки)
columns = [""] * KEY_LENGTH
for i, char in enumerate(ciphertext_no_spaces):
    columns[i % KEY_LENGTH] += char

# Функция для частотного анализа колонки
def frequency_analysis(text):
    counts = Counter(text)
    total = len(text)
    freq = {char: counts.get(char, 0) / total for char in RUSSIAN_ALPHABET}
    return freq, counts

# Анализируем каждую колонку
column_frequencies = []
column_counts = []
print("\n--- Частотный Анализ по Колонкам (Ключ=11) ---")
for i in range(KEY_LENGTH):
    print(f"\n--- Колонка {i+1} ---")
    freq, counts = frequency_analysis(columns[i])
    column_frequencies.append(freq)
    column_counts.append(counts)
    # Выведем топ-5 букв для каждой колонки
    sorted_counts = sorted(counts.items(), key=lambda item: item[1], reverse=True)
    print(f"  Текст (начало): {columns[i][:20]}...")
    print(f"  Топ-5 частот: {sorted_counts[:5]}")


--- Частотный Анализ по Колонкам (Ключ=11) ---

--- Колонка 1 ---
  Текст (начало): ДРСЯЮЫИРМИМЕНЬОСНМЧТ...
  Топ-5 частот: [('М', 6), ('Н', 5), ('С', 4), ('Д', 3), ('Ы', 3)]

--- Колонка 2 ---
  Текст (начало): АСВМТНГВДЛУПУТЦОВЫАЫ...
  Топ-5 частот: [('Т', 6), ('Е', 6), ('С', 5), ('Н', 5), ('А', 4)]

--- Колонка 3 ---
  Текст (начало): МЫЕДИДАКНДНЬЮКИЕММТМ...
  Топ-5 частот: [('М', 6), ('Д', 6), ('Н', 6), ('И', 5), ('К', 5)]

--- Колонка 4 ---
  Текст (начало): ИСДМЫТНТИАТКНПМНИЙУЖ...
  Топ-5 частот: [('Т', 8), ('Н', 6), ('И', 4), ('С', 4), ('Д', 4)]

--- Колонка 5 ---
  Текст (начало): НАТЫМЫХДГСИСХДКЫЛЫСА...
  Топ-5 частот: [('А', 7), ('Т', 6), ('Ы', 6), ('С', 5), ('Д', 4)]

--- Колонка 6 ---
  Текст (начало): АМКМЬМВАЕДИХНСКМЬНЬЛ...
  Топ-5 частот: [('М', 6), ('Ь', 5), ('Н', 5), ('Ы', 5), ('А', 4)]

--- Колонка 7 ---
  Текст (начало): МЦЧЮТРСМКПГЫУЯНОСУКШ...
  Топ-5 частот: [('М', 7), ('С', 7), ('Т', 4), ('К', 3), ('Н', 3)]

--- Колонка 8 ---
  Текст (начало): ТИВНКГОИНЬВССНОПОСТ

In [3]:
# Функция для расчета Хи-квадрат
def calculate_chi_squared(observed_counts, column_len):
    chi_squared = 0
    for i, char_expected in enumerate(RUSSIAN_ALPHABET):
        # Ожидаемое количество
        expected_count = RUSSIAN_FREQ_NORMALIZED.get(char_expected, 0) * column_len
        # Наблюдаемое количество (после гипотетического сдвига, которое мы будем имитировать)
        observed_count = observed_counts.get(char_expected, 0)

        # Избегаем деления на ноль, если ожидаемая частота очень мала
        if expected_count < 1e-5:
             # Если и наблюдаемая частота 0, вклад = 0, иначе - большой штраф
             if observed_count > 0:
                 chi_squared += 1000 # Большое значение
             continue # Иначе пропускаем

        chi_squared += (observed_count - expected_count)**2 / expected_count
    return chi_squared

# Функция для сдвига текста (дешифровка одним символом ключа)
def decrypt_char(char, key_char):
    if char not in RUSSIAN_ALPHABET or key_char not in RUSSIAN_ALPHABET:
        return char # Не шифруем/дешифруем неалфавитные символы
    char_idx = RUSSIAN_ALPHABET.find(char)
    key_idx = RUSSIAN_ALPHABET.find(key_char)
    decrypted_idx = (char_idx - key_idx + ALPHABET_LEN) % ALPHABET_LEN
    return RUSSIAN_ALPHABET[decrypted_idx]

# Находим лучшую букву ключа для каждой колонки
potential_key = ""
print("\n--- Определение Ключа (Метод Хи-Квадрат) ---")
for i in range(KEY_LENGTH):
    best_key_char = '?'
    min_chi_squared = float('inf')

    original_column_text = columns[i]
    column_len = len(original_column_text)

    # Пробуем каждую букву алфавита как возможную букву ключа для этой колонки
    for key_attempt_char in RUSSIAN_ALPHABET:
        # Дешифруем колонку этим ключом
        decrypted_column_attempt = "".join([decrypt_char(c, key_attempt_char) for c in original_column_text])
        # Считаем частоты в дешифрованной колонке
        observed_counts = Counter(decrypted_column_attempt)
        # Считаем Хи-квадрат
        chi_val = calculate_chi_squared(observed_counts, column_len)

        if chi_val < min_chi_squared:
            min_chi_squared = chi_val
            best_key_char = key_attempt_char

    print(f"Колонка {i+1}: Лучшая буква ключа = '{best_key_char}' (Chi^2 = {min_chi_squared:.2f})")
    potential_key += best_key_char

print(f"\nПредполагаемый ключ (длина {KEY_LENGTH}): {potential_key}")


--- Определение Ключа (Метод Хи-Квадрат) ---
Колонка 1: Лучшая буква ключа = 'А' (Chi^2 = 41.39)
Колонка 2: Лучшая буква ключа = 'А' (Chi^2 = 28.28)
Колонка 3: Лучшая буква ключа = 'А' (Chi^2 = 65.53)
Колонка 4: Лучшая буква ключа = 'А' (Chi^2 = 51.23)
Колонка 5: Лучшая буква ключа = 'А' (Chi^2 = 55.69)
Колонка 6: Лучшая буква ключа = 'А' (Chi^2 = 62.96)
Колонка 7: Лучшая буква ключа = 'А' (Chi^2 = 62.13)
Колонка 8: Лучшая буква ключа = 'А' (Chi^2 = 43.15)
Колонка 9: Лучшая буква ключа = 'А' (Chi^2 = 41.49)
Колонка 10: Лучшая буква ключа = 'А' (Chi^2 = 27.38)
Колонка 11: Лучшая буква ключа = 'А' (Chi^2 = 52.70)

Предполагаемый ключ (длина 11): ААААААААААА


In [5]:
# Функция дешифровки Виженера
def vigenere_decrypt(ciphertext, key):
    decrypted_text = ""
    key_len = len(key)
    key_index = 0
    for char in ciphertext: # Используем оригинальный текст С ПРОБЕЛАМИ
        if char == ' ': # Сохраняем пробелы
            decrypted_text += ' '
            continue # Не сдвигаем индекс ключа на пробелах

        # Обрабатываем только буквы из алфавита
        if char in RUSSIAN_ALPHABET:
             key_char = key[key_index % key_len]
             decrypted_char = decrypt_char(char, key_char)
             decrypted_text += decrypted_char
             key_index += 1 # Сдвигаем индекс ключа только для букв
        else:
             decrypted_text += char # Добавляем символы (цифры, #), если они есть, без изменений
             # key_index += 1 # Спорный момент - сдвигать ли ключ на не-буквах. Обычно нет.

    return decrypted_text

# Дешифруем консенсусный текст (С ПРОБЕЛАМИ!) найденным ключом
final_decrypted_text = vigenere_decrypt(ciphertext, potential_key) # Используем текст С пробелами

print("\n" + "="*30 + " Финальный Расшифрованный Текст " + "="*30)
print("\n" + final_decrypted_text)
print("\n" + "=" * (32 + len(" Финальный Расшифрованный Текст ")))


============================== Финальный Расшифрованный Текст ==============================

ДАМИНАМТОТИРСЫСАМЦИИЛ ЬСВЕДТКЧВНТИ ЯМДМЫМЮНЯМЦ ЮТИЫМ ЬТКТЧМЫН ДТЫМРГЗ ЕПИГАНХ ВСОЕГЬ Р ВКТДАМИ НКШМДНИ ГЕКНЖТАИЛ ДАСДП ЬСОЕМУНТИ ИГВКСОЕП ЬКСХЫСУС ХНУ ЮН ХНУСИЕТЬТКП ДСЯНКМЫОЦ ИМК КНОЬКМ СОЕНЫМОП ЬСЮНВМ ИЛ ЬСОЕНДМЫМ ЙЫНУС ЙЫМЧАТУС ЬКТДЛХТ ЫМЖАЛШ ДЛУСВ М ЧТЫНАМБИЛ ДАСДП ОСЮВНЫМ НЮЙГРГТЫТУКНЩНАС АТ ИСЧТ КГЖНЕПОЦ ЮН НЙОСЫЗЕАЗ ДТКАСОЕПШМИДСЫСДТОЫЗДЛ ОЫЛЕТ ЕС ЫНАТ ЕСЗЕРЫАМЕАН ЖНОЕСЕТ ЬТКТВНЖМ ГДТЫМЖТААСБ АН РМЫСЯМРЫСДОДЦЮП ЬСВВТКЧМДНТЕОЦ ДКГЖАГЗ АТГЕСИМИЛИМ ЬНЫПЯНИМ АНХМШ СЬТКНЕСКСДОРСКС ИЛ ДАСДП ЬСОЕМУАТИЕНБАЛ КНРТЕАЛШ ВДМУНЕТЫТБОИСЧТИ СЕЬКНДМЕП ЮН ДНИМ ОЬНОМЕТЫПАЛБ ЖТЫАСР ТОЫМ ДЛ ЬСЧТЫНТЕТ ДТКАГЕПОЦ ВСИСБ НН ОДТЕ ОСЫАЯН ВНКМЕ ЧМЮАП РНЧВСИГРСАТЯ ЬТКТВНЖМ

